In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [4]:
from torch.utils.data import DataLoader
from torchvision.transforms import transforms

In [5]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data",transform=transform,download=False,train=True)

testset = CIFAR10(root="./data",transform=transform,download=False,train=False)

c:\Users\Farhan Khan\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [6]:
trainloader=DataLoader(trainset,shuffle=True,batch_size=64)
testloader=DataLoader(testset,batch_size=64)


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layer=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )
        self.fc_layer=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        x=self.conv_layer(x)
        x=torch.flatten(x,1)
        x=self.fc_layer(x)

        return x


In [8]:
Model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(Model.parameters())

In [11]:
epochs=10
trainloss=[]
valloss=[]
for epoch in range(epochs):
    training_loss=0.0
    for images,labels in trainloader:
        optimizer.zero_grad()
        outputs=Model.forward(images)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()

        training_loss+=loss.item()
        trainloss.append(training_loss)
    Model.eval()
    val_running_loss=0.0
    for xa,ya in testloader:
        optimizer.zero_grad()
        output=Model.forward(xa)
        losses=criterion(output,ya)
        val_running_loss+=loss
        valloss.append(val_running_loss)

    print(f'Epochs={epoch+1}/{epochs} & train_loss ={training_loss/len(trainloader)} & test_loss={val_running_loss/len(testloader)}')



Epochs=1/10 & train_loss =0.14462250983699812 & test_loss=0.2692846953868866
Epochs=2/10 & train_loss =0.12147435341375734 & test_loss=0.006492805201560259
Epochs=3/10 & train_loss =0.1014144276935712 & test_loss=0.08419029414653778
Epochs=4/10 & train_loss =0.10109045831641406 & test_loss=0.250309020280838
Epochs=5/10 & train_loss =0.08373575262300184 & test_loss=0.07604247331619263
Epochs=6/10 & train_loss =0.07964499879275899 & test_loss=0.015150791965425014
Epochs=7/10 & train_loss =0.08637629808825881 & test_loss=0.03717326372861862
Epochs=8/10 & train_loss =0.07222058686434918 & test_loss=0.07679587602615356
Epochs=9/10 & train_loss =0.07254000822477558 & test_loss=0.0018368862802162766
Epochs=10/10 & train_loss =0.06303215443955787 & test_loss=0.25941887497901917


In [12]:
correct=0
total=0
Model.eval()
with torch.no_grad():
    for images,labels in testloader:
        output=Model.forward(images)
        _,predicted=torch.max(output,1)

        correct+=(predicted==labels).sum().item()
        total+=labels.size(0)
print(f'Accuracy={correct/total*100}')

Accuracy=74.65
